# Generador del tablero de proposiciones al PND 2018-2022

Este cuaderno de **Google Colab** toma el cuadro de proposiciones en Excel (plantilla `Proposiciones.xlsx` ya diligenciada), lo **valida**, y genera el sitio web del tablero listo para publicar en **GitHub Pages**.

**Qué hace, en orden**
1. Carga el Excel (se lo pide al ejecutar la celda 3).
2. Lo lee y ejecuta 25 controles de calidad. Si hay errores, se detiene y los muestra.
3. Enriquece los datos (posición Gobierno / Independiente / Oposición, favorabilidad, trazabilidad) y los exporta a `data/`.
4. Escribe los archivos del tablero (`index.html`, `assets/styles.css`, `assets/app.js`).
5. Muestra una vista previa en el cuaderno y descarga un **ZIP** con el repositorio completo.

> Los datos de ejemplo son **simulados** (nombres de congresistas ficticios). Para usar datos reales, diligencie el Excel con la misma estructura y conserve la hoja `Catalogos`.

**Uso:** menú *Entorno de ejecución → Ejecutar todo*.

## 1. Preparación

In [ ]:
%pip install -q openpyxl pandas
from pathlib import Path
import shutil, json
SITIO = Path("tablero-pnd-2018-2022")          # carpeta del repositorio que se va a generar
for sub in ("assets", "data", "notebooks"):
    (SITIO / sub).mkdir(parents=True, exist_ok=True)
print("Carpeta de salida:", SITIO.resolve())

## 2. Cargar el Excel
Seleccione `Proposiciones_PND_2018-2022_diligenciado.xlsx` (o su propio archivo con la misma estructura). Si ya está en la carpeta de trabajo del cuaderno, puede cancelar la ventana de carga.

In [ ]:
NOMBRE_XLSX = "Proposiciones_PND_2018-2022_diligenciado.xlsx"
try:
    from google.colab import files
    subidos = files.upload()
    if subidos:
        NOMBRE_XLSX = next(iter(subidos))
except ImportError:
    print("(Fuera de Colab: se usa el archivo de la carpeta de trabajo)")
RUTA_XLSX = Path(NOMBRE_XLSX)
assert RUTA_XLSX.exists(), f"No se encontró {RUTA_XLSX}. Cárguelo con la celda anterior."
print("Archivo:", RUTA_XLSX)

## 3. Leer el cuadro

In [ ]:
import json
import re
from datetime import date, datetime, time
from pathlib import Path

import openpyxl
import pandas as pd

FILA_ENCABEZADO = 4
FILA_INICIO = 5
N_COLUMNAS = 33  # A:AG de la plantilla

COLUMNAS = [
    "ID proposición", "Fecha recepción", "Hora", "Congresista autor", "Cámara / instancia",
    "Partido / bancada", "Región / circunscripción", "Coponentes / apoyos", "Tipo de proposición",
    "Artículo del proyecto afectado", "Eje líder", "Tema / subtema", "Texto original de la proposición",
    "Objeto / intención", "Justificación presentada", "Estado", "Concepto técnico", "Concepto jurídico",
    "Concepto fiscal", "Recomendación Gobierno", "Nivel de prioridad", "Riesgo jurídico",
    "Impacto fiscal estimado COP", "Fuente propuesta", "Entidad líder análisis", "Responsable seguimiento",
    "Fecha límite respuesta", "Reunión / sesión asociada", "Gaceta / radicado", "URL / fuente",
    "Observaciones", "Última actualización", "Versión",
]


def _a_fecha(v):
    if isinstance(v, datetime):
        return v.date()
    if isinstance(v, date):
        return v
    if isinstance(v, str) and v.strip():
        return datetime.strptime(v.strip()[:10], "%Y-%m-%d").date()
    return None


def _a_hora(v):
    if isinstance(v, datetime):
        return v.time().replace(second=0, microsecond=0)
    if isinstance(v, time):
        return v.replace(second=0, microsecond=0)
    if isinstance(v, str) and v.strip():
        h, m = v.strip().split(":")[:2]
        return time(int(h), int(m))
    return None


def leer_catalogos(wb):
    """Lee la hoja 'Catalogos': cada columna con encabezado en la fila 1 es una lista."""
    if "Catalogos" not in wb.sheetnames:
        raise ValueError("Falta la hoja 'Catalogos' (partidos, estados, tipos, entidades...).")
    ws = wb["Catalogos"]
    cat = {}
    for col in ws.iter_cols(min_row=1):
        enc = col[0].value
        if enc:
            cat[str(enc).strip()] = [c.value for c in col[1:] if c.value not in (None, "")]
    return cat


def leer_proposiciones(ruta):
    """Devuelve (df, catalogos). Las filas con 'Texto original' vacío se ignoran."""
    wb = openpyxl.load_workbook(ruta, data_only=True)
    ws = wb.worksheets[0]
    catalogos = leer_catalogos(wb)
    encabezados = [c.value for c in ws[FILA_ENCABEZADO]][:N_COLUMNAS]
    if encabezados != COLUMNAS:
        faltan = [c for c in COLUMNAS if c not in encabezados]
        raise ValueError(f"Los encabezados de la fila {FILA_ENCABEZADO} no coinciden con la plantilla. Faltan: {faltan}")
    filas = []
    for n_fila, fila in enumerate(ws.iter_rows(min_row=FILA_INICIO, max_col=N_COLUMNAS, values_only=True), start=FILA_INICIO):
        if fila[12] in (None, ""):
            continue
        filas.append(list(fila) + [n_fila])
    df = pd.DataFrame(filas, columns=COLUMNAS + ["_fila"])
    # El ID de la plantilla es una fórmula; si el archivo no tiene valor calculado, se reconstruye.
    df["ID proposición"] = [
        v if isinstance(v, str) and v.startswith("PROP-") else f"PROP-{f - FILA_ENCABEZADO:04d}"
        for v, f in zip(df["ID proposición"], df["_fila"])
    ]
    for c in ("Fecha recepción", "Fecha límite respuesta", "Última actualización"):
        df[c] = df[c].map(_a_fecha)
    df["Hora"] = df["Hora"].map(_a_hora)
    return df, catalogos

In [ ]:
df, catalogos = leer_proposiciones(RUTA_XLSX)
print(f"{len(df)} proposiciones leídas · {df['Congresista autor'].nunique()} congresistas · {df['Partido / bancada'].nunique()} partidos")
df[["ID proposición", "Fecha recepción", "Congresista autor", "Partido / bancada", "Tipo de proposición", "Estado"]].head(10)

## 4. Validar el cuadro
Se detiene si algún control falla (ERROR). Las ADVERTENCIAS no detienen el proceso, pero conviene revisarlas.

In [ ]:
FECHA_RADICACION = date(2019, 2, 6)   # radicación del proyecto
FECHA_SANCION = date(2019, 5, 25)     # Ley 1955 de 2019

PREFIJOS_TIPO = {
    "Modificatoria": ("Modifíquese",),
    "Aditiva (artículo nuevo)": ("Adiciónese", "Restitúyanse"),
    "Supresiva": ("Suprímase",),
    "Sustitutiva": ("Sustitúyase",),
    "Aclaratoria / de forma": ("Aclárese",),
}
ESTADOS_FAVORABLES = {"Acogida total", "Aprobada"}
ESTADOS_DESFAVORABLES = {"No acogida", "Negada"}


def etapa_de(fecha):
    """Etapa del trámite según la fecha de recepción (calendario real del PND 2018-2022)."""
    if fecha <= date(2019, 3, 14):
        return "SUB"   # subcomisiones de ponentes
    if fecha <= date(2019, 3, 22):
        return "PD"    # primer debate, comisiones económicas conjuntas
    return "PL"        # plenarias


def validar(df, cat, n_esperado=None):
    """Aplica los controles de calidad. Devuelve lista de dicts: control, estado, detalle."""
    res = []

    def reg(control, ok, detalle="", nivel="ERROR"):
        res.append({"control": control, "estado": "OK" if ok else nivel, "detalle": detalle if not ok else (detalle or "Sin hallazgos")})

    if n_esperado is not None:
        reg("Número de registros", len(df) == n_esperado, f"Se esperaban {n_esperado} y hay {len(df)}")
    else:
        reg("Número de registros", len(df) > 0, f"{len(df)} registros")

    # 1. Campos obligatorios
    oblig = [c for c in COLUMNAS if c != "ID proposición"]
    vacios = {}
    for c in oblig:
        s = df[c]
        n = int(s.isna().sum() + ((s.astype(str).str.strip() == "") & s.notna()).sum())
        if n:
            vacios[c] = n
    reg("Campos obligatorios diligenciados", not vacios, f"Vacíos por columna: {vacios}")

    # 2. ID
    ids = df["ID proposición"]
    reg("ID único y con formato PROP-0000", ids.is_unique and ids.str.fullmatch(r"PROP-\d{4}").all(), "IDs repetidos o con formato inválido")

    # 3. Dominios (catálogos)
    dominios = [
        ("Cámara / instancia", "Cámara"), ("Partido / bancada", "Partido"), ("Tipo de proposición", "Tipo de proposición"),
        ("Eje líder", "Eje líder"), ("Estado", "Estado"), ("Concepto técnico", "Concepto"),
        ("Concepto jurídico", "Concepto"), ("Concepto fiscal", "Concepto"), ("Recomendación Gobierno", "Recomendación Gobierno"),
        ("Nivel de prioridad", "Nivel de prioridad"), ("Riesgo jurídico", "Riesgo jurídico"),
        ("Fuente propuesta", "Fuente propuesta"), ("Entidad líder análisis", "Entidad"),
    ]
    malos = {}
    for col, cat_nombre in dominios:
        fuera = sorted(set(df[col].dropna()) - set(cat.get(cat_nombre, [])))
        if fuera:
            malos[col] = fuera
    reg("Valores dentro de los catálogos", not malos, f"Fuera de catálogo: {malos}")

    # 4. Fechas
    f_rec, f_lim, f_upd = df["Fecha recepción"], df["Fecha límite respuesta"], df["Última actualización"]
    rango = (f_rec < FECHA_RADICACION) | (f_rec > FECHA_SANCION)
    reg("Fecha de recepción dentro del trámite (6-feb a 25-may-2019)", not rango.any(), f"Filas: {df.loc[rango, 'ID proposición'].tolist()}")
    reg("Fecha límite de respuesta ≥ fecha de recepción", (f_lim >= f_rec).all(), f"Filas: {df.loc[f_lim < f_rec, 'ID proposición'].tolist()}")
    reg("Última actualización ≥ fecha de recepción y ≤ sanción", ((f_upd >= f_rec) & (f_upd <= FECHA_SANCION)).all(),
        f"Filas: {df.loc[(f_upd < f_rec) | (f_upd > FECHA_SANCION), 'ID proposición'].tolist()}")
    habil = f_rec.map(lambda d: d.weekday() < 5)
    reg("Fechas de recepción en días hábiles", habil.all(), f"Filas: {df.loc[~habil, 'ID proposición'].tolist()}", nivel="ADVERTENCIA")
    horas_ok = df["Hora"].map(lambda t: t is not None and time(7, 0) <= t <= time(21, 0))
    reg("Hora de recepción entre 07:00 y 21:00", horas_ok.all(), f"Filas: {df.loc[~horas_ok, 'ID proposición'].tolist()}", nivel="ADVERTENCIA")
    orden = df.sort_values("_fila")
    marca = [(d, h) for d, h in zip(orden["Fecha recepción"], orden["Hora"])]
    reg("Registros en orden cronológico de recepción", marca == sorted(marca), "El orden de filas no sigue la fecha/hora", nivel="ADVERTENCIA")

    # 5. Congresista <-> cámara / partido / circunscripción
    pos = dict(zip(cat.get("Partido", []), cat.get("Posición frente al Gobierno", [])))
    incons = df.groupby("Congresista autor")[["Cámara / instancia", "Partido / bancada", "Región / circunscripción"]].nunique()
    reg("Cada congresista conserva cámara, partido y circunscripción", (incons <= 1).all().all(),
        f"Inconsistentes: {incons[(incons > 1).any(axis=1)].index.tolist()}")
    sin_pos = sorted(set(df["Partido / bancada"]) - set(pos))
    reg("Todo partido tiene posición Gobierno / Independiente / Oposición", not sin_pos, f"Sin posición: {sin_pos}")
    sen = df["Cámara / instancia"] == "Senado de la República"
    circ_ok = (~sen) | df["Región / circunscripción"].str.startswith("Nacional")
    reg("Senadores con circunscripción nacional", circ_ok.all(), f"Filas: {df.loc[~circ_ok, 'ID proposición'].tolist()}")

    # 6. Texto de la proposición vs. tipo
    def prefijo_ok(r):
        pref = PREFIJOS_TIPO.get(r["Tipo de proposición"])
        return pref is None or str(r["Texto original de la proposición"]).startswith(pref)
    okp = df.apply(prefijo_ok, axis=1)
    reg("Redacción de la proposición coherente con su tipo", okp.all(), f"Filas: {df.loc[~okp, 'ID proposición'].tolist()}")
    reg("Textos sin duplicados", df["Texto original de la proposición"].is_unique, "Hay textos repetidos")
    largo = df["Texto original de la proposición"].astype(str).str.len() >= 40
    reg("Texto de la proposición con extensión mínima (40 caracteres)", largo.all(), f"Filas: {df.loc[~largo, 'ID proposición'].tolist()}")

    # 7. Estado vs. etapa del trámite
    et = f_rec.map(etapa_de)
    permitidos = {"SUB": {"Acogida total", "Acogida parcial", "No acogida", "Retirada"},
                  "PD": {"Aprobada", "Negada", "Retirada"},
                  "PL": {"Aprobada", "Negada", "Acogida parcial", "En análisis", "Retirada"}}
    mal_estado = [i for i, e, s in zip(df["ID proposición"], et, df["Estado"]) if s not in permitidos[e]]
    reg("Estado coherente con la etapa del trámite", not mal_estado, f"Filas: {mal_estado}")

    # 8. Recomendación del Gobierno vs. estado (excepciones deben documentarse)
    exc = []
    for i, s, rec, obs in zip(df["ID proposición"], df["Estado"], df["Recomendación Gobierno"], df["Observaciones"]):
        contra = (s in ESTADOS_FAVORABLES and rec == "No acoger") or (s in ESTADOS_DESFAVORABLES and rec == "Acoger")
        if contra and "pese a" not in str(obs).lower():
            exc.append(i)
    reg("Estado contrario a la recomendación del Gobierno está documentado en Observaciones", not exc, f"Filas sin justificar: {exc}")

    # 9. Conceptos vs. recomendación
    c3 = df[["Concepto técnico", "Concepto jurídico", "Concepto fiscal"]]
    hay_desf = (c3 == "Desfavorable").any(axis=1)
    pendientes = (c3 == "Pendiente").all(axis=1)
    rec = df["Recomendación Gobierno"]
    err_a = df.loc[(rec == "Acoger") & hay_desf, "ID proposición"].tolist()
    err_n = df.loc[(rec == "No acoger") & ~hay_desf & ~pendientes, "ID proposición"].tolist()
    reg("Recomendación 'Acoger' sin conceptos desfavorables", not err_a, f"Filas: {err_a}")
    reg("Recomendación 'No acoger' respaldada por al menos un concepto desfavorable", not err_n, f"Filas: {err_n}")
    en_analisis = df["Estado"] == "En análisis"
    reg("Proposiciones 'En análisis' con conceptos pendientes", (pendientes | ~en_analisis).all(), "Hay 'En análisis' con conceptos emitidos")

    # 10. Impacto fiscal
    imp = pd.to_numeric(df["Impacto fiscal estimado COP"], errors="coerce")
    reg("Impacto fiscal numérico", imp.notna().all(), f"Filas: {df.loc[imp.isna(), 'ID proposición'].tolist()}")
    reg("Impacto fiscal en rango razonable (|x| ≤ 5 billones COP)", (imp.abs() <= 5e12).all(), "Valores fuera de rango")
    sin_cuantia = df.loc[(imp > 500e9) & (df["Concepto fiscal"] == "Favorable"), "ID proposición"].tolist()
    reg("Concepto fiscal 'Favorable' solo con impactos ≤ 500 mil millones", not sin_cuantia, f"Filas: {sin_cuantia}")

    # 11. Marca de datos simulados
    sim = df["Observaciones"].astype(str).str.startswith("[SIMULADO]")
    reg("Registros marcados como [SIMULADO] en Observaciones", sim.all(), "Hay registros sin la marca", nivel="ADVERTENCIA")

    return res


def resumen_validacion(res):
    n_err = sum(r["estado"] == "ERROR" for r in res)
    n_adv = sum(r["estado"] == "ADVERTENCIA" for r in res)
    return n_err, n_adv

In [ ]:
resultados = validar(df, catalogos)          # use validar(df, catalogos, n_esperado=50) para exigir un número exacto
tabla = pd.DataFrame(resultados)
tabla["estado"] = tabla["estado"].map({"OK": "✅ OK", "ADVERTENCIA": "⚠️ ADVERTENCIA", "ERROR": "❌ ERROR"})
pd.set_option("display.max_colwidth", 120, "display.width", 200)
display(tabla)
n_err, n_adv = resumen_validacion(resultados)
print(f"Errores: {n_err} · Advertencias: {n_adv}")
if n_err:
    raise SystemExit("Corrija los errores en el Excel y vuelva a ejecutar el cuaderno.")

## 5. Enriquecer y exportar los datos del tablero
Los hitos del trámite (círculos numerados de la línea de tiempo) se definen aquí; edítelos si necesita otros.

In [ ]:
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]


def _fmt(d):
    return f"{d.day:02d}-{MESES[d.month - 1]}-{d.year}"


def construir_registros(df, cat):
    """Convierte el DataFrame validado en la lista de registros que consume el tablero."""
    pos = dict(zip(cat["Partido"], cat["Posición frente al Gobierno"]))
    fav = dict(zip(cat["Estado"], cat["Favorabilidad"]))
    sigla = dict(zip(cat["Entidad"], cat["Sigla entidad"]))
    registros = []
    for _, r in df.iterrows():
        estado = r["Estado"]
        registros.append({
            "id": r["ID proposición"],
            "fecha": r["Fecha recepción"].isoformat(),
            "hora": r["Hora"].strftime("%H:%M"),
            "congresista": r["Congresista autor"],
            "camara": r["Cámara / instancia"],
            "partido": r["Partido / bancada"],
            "posicion": pos[r["Partido / bancada"]],
            "circunscripcion": r["Región / circunscripción"],
            "apoyos": r["Coponentes / apoyos"],
            "tipo": r["Tipo de proposición"],
            "articulo": r["Artículo del proyecto afectado"],
            "eje": r["Eje líder"],
            "tema": r["Tema / subtema"],
            "texto": r["Texto original de la proposición"],
            "objeto": r["Objeto / intención"],
            "justificacion": r["Justificación presentada"],
            "estado": estado,
            "favorabilidad": fav[estado],
            "c_tecnico": r["Concepto técnico"],
            "c_juridico": r["Concepto jurídico"],
            "c_fiscal": r["Concepto fiscal"],
            "recomendacion": r["Recomendación Gobierno"],
            "prioridad": r["Nivel de prioridad"],
            "riesgo": r["Riesgo jurídico"],
            "impacto": int(r["Impacto fiscal estimado COP"]),
            "fuente": r["Fuente propuesta"],
            "entidad": r["Entidad líder análisis"],
            "entidad_corta": sigla[r["Entidad líder análisis"]],
            "responsable": r["Responsable seguimiento"],
            "fecha_limite": r["Fecha límite respuesta"].isoformat(),
            "sesion": r["Reunión / sesión asociada"],
            "radicado": r["Gaceta / radicado"],
            "url": r["URL / fuente"],
            "observaciones": r["Observaciones"],
            "ultima_actualizacion": r["Última actualización"].isoformat(),
            "version": int(r["Versión"]),
            "etapa": etapa_de(r["Fecha recepción"]),
            "trazabilidad": [
                {"fecha": r["Fecha recepción"].isoformat(), "hito": "Recepción",
                 "detalle": f"{r['Reunión / sesión asociada']} · {r['Hora'].strftime('%H:%M')} h · {r['Gaceta / radicado']}"},
                {"fecha": r["Fecha límite respuesta"].isoformat(), "hito": "Fecha límite de respuesta del Gobierno",
                 "detalle": f"Recomendación: {r['Recomendación Gobierno']} · {r['Entidad líder análisis']}"},
                {"fecha": r["Última actualización"].isoformat(), "hito": f"Estado: {estado}",
                 "detalle": f"Versión {int(r['Versión'])} · conceptos técnico/jurídico/fiscal: "
                            f"{r['Concepto técnico']} / {r['Concepto jurídico']} / {r['Concepto fiscal']}"},
            ],
        })
    return registros


def exportar_datos(df, cat, hitos, carpeta_datos):
    """Escribe data/proposiciones.json y data/proposiciones.js (este último permite abrir index.html sin servidor)."""
    carpeta = Path(carpeta_datos)
    carpeta.mkdir(parents=True, exist_ok=True)
    registros = construir_registros(df, cat)
    paquete = {
        "meta": {
            "titulo": "Plan Nacional de Desarrollo 2018-2022 «Pacto por Colombia, Pacto por la Equidad»",
            "norma": "Ley 1955 de 2019 · Proyecto de Ley 311/2019 Cámara – 227/2019 Senado",
            "generado": datetime.now().strftime("%Y-%m-%d %H:%M"),
            "n_registros": len(registros),
            "simulado": True,
            "orden_favorabilidad": ["Favorable", "Parcialmente favorable", "Desfavorable", "Pendiente", "Retirada"],
        },
        "hitos": hitos,
        "registros": registros,
    }
    (carpeta / "proposiciones.json").write_text(json.dumps(paquete, ensure_ascii=False, indent=1), encoding="utf-8")
    (carpeta / "proposiciones.js").write_text(
        "// Generado automáticamente por el notebook de Colab. No editar a mano.\n"
        "window.PND_DATA = " + json.dumps(paquete, ensure_ascii=False) + ";\n", encoding="utf-8")
    return paquete

In [ ]:
HITOS = [
  {
    "fecha": "2019-02-06",
    "etiqueta": "Radicación del proyecto",
    "detalle": "Proyecto de Ley 311/2019 Cámara – 227/2019 Senado"
  },
  {
    "fecha": "2019-03-14",
    "etiqueta": "Subcomisión de artículos nuevos",
    "detalle": "Ponentes estudian proposiciones de nuevos artículos"
  },
  {
    "fecha": "2019-03-19",
    "etiqueta": "Ponencias primer debate",
    "detalle": "Radicación de informes de ponencia"
  },
  {
    "fecha": "2019-03-22",
    "etiqueta": "Primer debate",
    "detalle": "Comisiones Económicas Conjuntas aprueban el texto"
  },
  {
    "fecha": "2019-04-26",
    "etiqueta": "Ponencias segundo debate",
    "detalle": "Informes para plenarias"
  },
  {
    "fecha": "2019-05-02",
    "etiqueta": "Plenarias",
    "detalle": "Aprobación en Cámara y Senado"
  },
  {
    "fecha": "2019-05-25",
    "etiqueta": "Sanción",
    "detalle": "Ley 1955 de 2019"
  }
]

paquete = exportar_datos(df, catalogos, HITOS, SITIO / "data")
shutil.copy(RUTA_XLSX, SITIO / "data" / RUTA_XLSX.name)
print(f"{paquete['meta']['n_registros']} registros exportados a {SITIO / 'data'}")
pd.DataFrame(paquete["registros"])[["favorabilidad", "posicion", "camara"]].apply(lambda s: s.value_counts()).T

## 6. Escribir los archivos del sitio
Tres celdas con el código de la página. Puede modificarlas (colores, textos) antes de ejecutarlas.

In [ ]:
%%writefile tablero-pnd-2018-2022/index.html
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Tablero de proposiciones al Plan Nacional de Desarrollo 2018-2022</title>
  <meta name="description" content="Tablero de seguimiento de proposiciones al Plan Nacional de Desarrollo 2018-2022 (Ley 1955 de 2019): trazabilidad cronológica, filtros por congresista, partido, posición frente al Gobierno, cámara, entidad y tipo de proposición. Datos simulados con fines demostrativos.">
  <meta name="color-scheme" content="light">
  <link rel="icon" href="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 32 32'%3E%3Crect width='32' height='32' rx='6' fill='%23102A43'/%3E%3Crect x='6' y='17' width='5' height='9' fill='%23fcd116'/%3E%3Crect x='13.5' y='11' width='5' height='15' fill='%23ffffff'/%3E%3Crect x='21' y='6' width='5' height='20' fill='%23ce1126'/%3E%3C/svg%3E">
  <link rel="stylesheet" href="assets/styles.css">
</head>
<body>
  <a class="skip" href="#contenido">Ir al contenido</a>
  <div class="flag" aria-hidden="true"><span></span><span></span><span></span></div>

  <header class="masthead">
    <div class="wrap masthead-in">
      <div>
        <h1>Proposiciones al Plan Nacional de Desarrollo 2018-2022</h1>
        <p class="sub">Pacto por Colombia, Pacto por la Equidad · Ley 1955 de 2019 · Proyecto de Ley 311/2019 Cámara – 227/2019 Senado</p>
      </div>
      <p class="pill">Datos simulados</p>
    </div>
  </header>

  <div class="aviso" role="note">
    <div class="wrap">
      <svg width="16" height="16" viewBox="0 0 16 16" aria-hidden="true"><circle cx="8" cy="8" r="7" fill="none" stroke="currentColor" stroke-width="1.5"/><path d="M8 7v4.2M8 4.6v.1" stroke="currentColor" stroke-width="1.6" stroke-linecap="round"/></svg>
      <p>Este tablero es demostrativo: los registros son simulados y los nombres de los congresistas, ficticios. El calendario del trámite y la posición de los partidos frente al Gobierno provienen de información pública.</p>
    </div>
  </div>

  <div class="filtros-wrap">
    <form id="filtros" class="wrap filtros" aria-label="Filtros del tablero" onsubmit="return false">
      <div class="f">
        <label for="f-congresista">Congresista</label>
        <select id="f-congresista"></select>
      </div>
      <div class="f">
        <label for="f-partido">Partido</label>
        <select id="f-partido"></select>
      </div>
      <fieldset class="f seg" id="f-posicion">
        <legend>Gobierno / Oposición</legend>
        <div class="segs" role="group"></div>
      </fieldset>
      <fieldset class="f seg" id="f-camara">
        <legend>Cámara</legend>
        <div class="segs" role="group"></div>
      </fieldset>
      <div class="f">
        <label for="f-entidad">Entidad líder del análisis</label>
        <select id="f-entidad"></select>
      </div>
      <div class="f">
        <label for="f-tipo">Tipo de proposición</label>
        <select id="f-tipo"></select>
      </div>
      <div class="f f-fecha">
        <label for="f-desde">Recibida desde</label>
        <input type="date" id="f-desde">
      </div>
      <div class="f f-fecha">
        <label for="f-hasta">Recibida hasta</label>
        <input type="date" id="f-hasta">
      </div>
      <div class="f acciones">
        <button type="button" id="limpiar" class="btn">Limpiar filtros</button>
        <button type="button" id="exportar" class="btn btn-sec">Descargar CSV</button>
      </div>
      <div class="estado-fila">
        <p id="estado-filtros" class="estado" aria-live="polite"></p>
        <ul id="chips" class="chips" aria-label="Filtros activos"></ul>
      </div>
    </form>
  </div>

  <main id="contenido" class="wrap">
    <section id="resumen" aria-labelledby="t-resumen">
      <h2 id="t-resumen">Conteo de favorabilidad</h2>
      <p class="desc">La favorabilidad resume el estado de cada proposición: favorable (acogida total o aprobada), parcialmente favorable (acogida parcial), desfavorable (no acogida o negada), pendiente (en análisis) o retirada. Seleccione una categoría para filtrar.</p>
      <div class="kpis" id="kpis"></div>
    </section>

    <section id="trazabilidad" aria-labelledby="t-traza">
      <h2 id="t-traza">Trazabilidad cronológica</h2>
      <p class="desc">Proposiciones recibidas por semana a lo largo del trámite del proyecto. Los círculos numerados marcan los hitos del proceso legislativo. Seleccione una semana para filtrar por fecha de recepción.</p>
      <article class="card" id="card-timeline">
        <header class="card-h">
          <div>
            <p class="cap">Número de proposiciones recibidas por semana, según favorabilidad</p>
            <div class="legend" data-legend></div>
          </div>
          <button type="button" class="link" id="vista-timeline" aria-pressed="false">Ver tabla</button>
        </header>
        <div id="timeline" class="timeline"></div>
        <div id="timeline-tabla" class="tabla-alt" hidden></div>
        <ol id="hitos" class="hitos"></ol>
      </article>

      <h3 class="h3-lista" id="t-lista">Registro cronológico de proposiciones</h3>
      <p class="desc">Ordenado por fecha y hora de recepción. Despliegue una fila para ver la trazabilidad de la proposición, su texto, conceptos y el impacto fiscal estimado.</p>
      <div class="tabla-wrap">
        <table class="registro" id="registro" aria-describedby="t-lista">
          <caption class="sr-only">Registro cronológico de proposiciones al Plan Nacional de Desarrollo</caption>
          <thead>
            <tr>
              <th scope="col"><span class="sr-only">Detalle</span></th>
              <th scope="col">Recepción</th>
              <th scope="col">Proposición</th>
              <th scope="col">Congresista</th>
              <th scope="col">Posición</th>
              <th scope="col">Artículo afectado</th>
              <th scope="col">Tipo</th>
              <th scope="col">Estado</th>
            </tr>
          </thead>
          <tbody id="registro-cuerpo"></tbody>
        </table>
      </div>
    </section>

    <section id="desgloses" aria-labelledby="t-desg">
      <h2 id="t-desg">Favorabilidad según congresista, partido, posición, cámara, entidad y tipo</h2>
      <p class="desc">Cada barra muestra cuántas proposiciones hay en la categoría y cómo se reparten por favorabilidad, dentro de los filtros activos. Seleccione una barra para filtrar.</p>
      <div class="legend legend-sec" data-legend></div>
      <div class="grid2" id="desgloses-grid"></div>
    </section>
  </main>

  <footer class="pie">
    <div class="wrap">
      <p><strong>Nota metodológica.</strong> Los registros (proposiciones, conceptos, impactos fiscales y radicados) son simulados con fines demostrativos y los nombres de los congresistas son ficticios. El calendario del trámite (radicación del proyecto el 6-feb-2019, primer debate en comisiones económicas conjuntas el 22-mar-2019, plenarias el 2-may-2019 y sanción el 25-may-2019) y la posición de los partidos frente al Gobierno (Estatuto de la Oposición, Ley 1909 de 2018) se basan en información pública. Los enlaces de cada registro apuntan a documentos de referencia del trámite, no a las proposiciones simuladas.</p>
      <p>Este sitio no es un canal oficial del Congreso de la República ni del Departamento Nacional de Planeación. <span id="generado"></span></p>
    </div>
  </footer>

  <div id="tip" class="tip" role="tooltip" hidden></div>
  <noscript><p class="wrap" style="padding:1rem 0">Este tablero requiere JavaScript para mostrar los filtros y los gráficos.</p></noscript>
  <script src="data/proposiciones.js"></script>
  <script src="assets/app.js"></script>
</body>
</html>

In [ ]:
%%writefile tablero-pnd-2018-2022/assets/styles.css
/* Tablero de proposiciones al PND 2018-2022 — estilos (sin dependencias externas) */
:root {
  color-scheme: light;
  --navy: #102a43;
  --navy-2: #1d3d5c;
  --teal: #0b7a75;
  --teal-dark: #085d59;
  --teal-soft: #eaf3f2;
  --bg: #f2f4f6;
  --surface: #ffffff;
  --ink: #14212d;
  --ink-2: #44525f;
  --muted: #5b6774;
  --line: #dde2e7;
  --grid: #e6e9ec;
  --axis: #c3c9cf;
  --focus: #0b7a75;
  /* Favorabilidad: rampa divergente azul ↔ rojo con grises neutros */
  --fav-1: #256abf; /* Favorable */
  --fav-2: #86b6ef; /* Parcialmente favorable */
  --fav-3: #e34948; /* Desfavorable */
  --fav-4: #898781; /* Pendiente */
  --fav-5: #c3c2b7; /* Retirada */
  --font: system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
}

*, *::before, *::after { box-sizing: border-box; }
html { -webkit-text-size-adjust: 100%; scroll-padding-top: 9rem; }
body { margin: 0; background: var(--bg); color: var(--ink); font: 15px/1.5 var(--font); }
h1, h2, h3, p, ul, ol { margin: 0; }
button, select, input { font: inherit; color: inherit; }
.wrap { width: min(1240px, 100% - 2rem); margin-inline: auto; }
.sr-only { position: absolute; width: 1px; height: 1px; overflow: hidden; clip: rect(0 0 0 0); white-space: nowrap; }
.skip { position: absolute; left: .5rem; top: -4rem; background: var(--navy); color: #fff; padding: .5rem .75rem; z-index: 100; border-radius: 4px; }
.skip:focus { top: .5rem; }
:focus-visible { outline: 2px solid var(--focus); outline-offset: 2px; }

/* Franja con los colores de la bandera (proporción 2:1:1) */
.flag { display: flex; height: 6px; }
.flag span:nth-child(1) { flex: 2; background: #fcd116; }
.flag span:nth-child(2) { flex: 1; background: #003893; }
.flag span:nth-child(3) { flex: 1; background: #ce1126; }

/* Encabezado */
.masthead { background: var(--navy); color: #fff; }
.masthead-in { display: flex; gap: 1.5rem; justify-content: space-between; align-items: flex-start; padding: 1.6rem 0 1.5rem; }
.masthead h1 { font-size: clamp(1.45rem, 2.4vw + .6rem, 2.1rem); line-height: 1.15; font-weight: 700; letter-spacing: -.01em; max-width: 24em; }
.masthead .sub { margin-top: .6rem; color: #c6d3df; max-width: 62em; font-size: .95rem; }
.pill { flex: none; border: 1px solid rgba(255,255,255,.4); border-radius: 999px; padding: .25rem .8rem; font-size: .85rem; color: #fff; white-space: nowrap; }
.aviso { background: #fff3d1; border-bottom: 1px solid #ecd48a; color: #5a4300; }
.aviso .wrap { display: flex; gap: .6rem; align-items: flex-start; padding: .55rem 0; font-size: .88rem; }
.aviso svg { flex: none; margin-top: .2rem; }

/* Filtros */
.filtros-wrap { background: var(--surface); border-bottom: 1px solid var(--line); z-index: 30; }
.filtros { display: flex; flex-wrap: wrap; gap: .7rem 1rem; align-items: flex-end; padding: .8rem 0; }
.f { display: flex; flex-direction: column; gap: .2rem; min-width: 0; border: 0; padding: 0; margin: 0; }
.f label, .f legend { font-size: .78rem; font-weight: 600; color: var(--ink-2); padding: 0; line-height: 1.2; display: block; float: none; }
.f legend { margin-bottom: .2rem; }
.f.seg { gap: 0; }
.cap { font-size: .84rem; font-weight: 600; color: var(--ink-2); margin-bottom: .35rem; }
select, input[type="date"] { height: 36px; border: 1px solid #aeb8c2; border-radius: 6px; background: #fff; padding: 0 .55rem; min-width: 0; }
.f select { width: 15.5rem; max-width: 100%; }
#f-tipo, #f-entidad { width: 14rem; }
.f-fecha input { width: 9.6rem; }
select:hover, input[type="date"]:hover { border-color: var(--ink-2); }
.segs { display: inline-flex; }
.segs button { height: 36px; padding: 0 .8rem; background: #fff; border: 1px solid #aeb8c2; margin-left: -1px; cursor: pointer; white-space: nowrap; }
.segs button:first-child { border-radius: 6px 0 0 6px; margin-left: 0; }
.segs button:last-child { border-radius: 0 6px 6px 0; }
.segs button:hover { background: var(--teal-soft); }
.segs button[aria-pressed="true"] { background: var(--navy); border-color: var(--navy); color: #fff; position: relative; z-index: 1; }
.acciones { flex-direction: row; gap: .5rem; margin-left: auto; }
.btn { height: 36px; padding: 0 .9rem; border-radius: 6px; border: 1px solid var(--teal); background: var(--teal); color: #fff; font-weight: 600; cursor: pointer; }
.btn:hover { background: var(--teal-dark); }
.btn-sec { background: #fff; color: var(--teal-dark); }
.btn-sec:hover { background: var(--teal-soft); }
.estado-fila { flex-basis: 100%; display: flex; flex-wrap: wrap; align-items: center; gap: .4rem .8rem; min-height: 1.6rem; }
.estado { font-size: .88rem; color: var(--ink-2); }
.estado strong { color: var(--ink); }
.chips { list-style: none; padding: 0; display: flex; flex-wrap: wrap; gap: .35rem; }
.chips button { display: inline-flex; align-items: center; gap: .35rem; background: var(--teal-soft); border: 1px solid #b9d9d6; color: var(--teal-dark); border-radius: 999px; padding: .1rem .35rem .1rem .65rem; font-size: .82rem; cursor: pointer; }
.chips button:hover { background: #d6ebe8; }
.chips .x { font-size: 1rem; line-height: 1; padding: 0 .25rem; }

/* Secciones */
main { padding-bottom: 2rem; }
section { margin-top: 2.2rem; }
h2 { font-size: 1.3rem; line-height: 1.25; font-weight: 700; color: var(--navy); }
h3 { font-size: 1.02rem; font-weight: 700; color: var(--navy); }
.desc { margin-top: .35rem; max-width: 72ch; color: var(--ink-2); font-size: .92rem; }
.h3-lista { margin-top: 1.8rem; font-size: 1.1rem; }

.card { background: var(--surface); border: 1px solid var(--line); border-radius: 8px; padding: 1rem 1.1rem; margin-top: .9rem; }
.card-h { display: flex; justify-content: space-between; gap: 1rem; align-items: flex-start; margin-bottom: .5rem; }
.card-h h3 { font-size: 1rem; }
.link { background: none; border: 0; color: var(--teal-dark); text-decoration: underline; text-underline-offset: 2px; cursor: pointer; padding: .15rem .25rem; font-size: .85rem; flex: none; }
.link:hover { color: var(--navy); }

/* Leyenda */
.legend { display: flex; flex-wrap: wrap; gap: .3rem 1.1rem; font-size: .84rem; color: var(--ink-2); }
.legend-sec { margin-top: .9rem; }
.legend span { display: inline-flex; align-items: center; gap: .4rem; }
.legend i { width: 12px; height: 12px; border-radius: 2px; display: inline-block; }

/* KPI */
.kpis { display: grid; grid-template-columns: 1.5fr repeat(5, minmax(0, 1fr)); gap: .75rem; margin-top: .9rem; }
.hero { background: var(--navy); color: #fff; border-radius: 8px; padding: 1rem 1.2rem; display: flex; flex-direction: column; justify-content: space-between; gap: .6rem; min-height: 8.2rem; }
.hero .v { font-size: 3.6rem; line-height: 1; font-weight: 700; letter-spacing: -.02em; }
.hero .l { font-size: .95rem; color: #d3deea; }
.hero .s { font-size: .82rem; color: #b6c6d6; border-top: 1px solid rgba(255,255,255,.2); padding-top: .5rem; }
.tile { text-align: left; background: var(--surface); border: 1px solid var(--line); border-left: 5px solid var(--c); border-radius: 8px; padding: .8rem .9rem; display: flex; flex-direction: column; gap: .2rem; cursor: pointer; }
.tile:hover { border-color: var(--c); background: #fbfcfd; }
.tile[aria-pressed="true"] { outline: 2px solid var(--navy); outline-offset: 1px; background: var(--teal-soft); }
.tile .v { font-size: 2rem; line-height: 1.05; font-weight: 700; }
.tile .l { font-weight: 600; font-size: .9rem; line-height: 1.25; }
.tile .p { font-size: .82rem; color: var(--muted); }

/* Línea de tiempo (SVG) */
.timeline { width: 100%; overflow-x: auto; }
.timeline svg { display: block; }
.timeline text { font-family: var(--font); fill: var(--muted); font-size: 11.5px; }
.timeline .num { fill: #fff; font-weight: 700; font-size: 11px; }
.timeline .ygrid { stroke: var(--grid); stroke-width: 1; }
.timeline .base { stroke: var(--axis); stroke-width: 1; }
.timeline .hito-l { stroke: #9aa6b2; stroke-width: 1; }
.timeline .hit { fill: transparent; cursor: pointer; }
.timeline .col:hover .wk-bg, .timeline .col:focus .wk-bg, .timeline .col.sel .wk-bg { fill: var(--teal-soft); }
.timeline .wk-bg { fill: transparent; }
.timeline .col:focus { outline: none; }
.timeline .col:focus-visible .wk-bg { stroke: var(--focus); stroke-width: 2; }
.timeline .hito-c { cursor: pointer; }
.timeline .hito-c:focus { outline: none; }
.timeline .hito-c:focus-visible circle, .timeline .hito-c:hover circle { stroke: var(--teal); stroke-width: 2.5; }
.timeline .empty { font-size: 14px; fill: var(--ink-2); }
.hitos { list-style: none; padding: 0; margin-top: .7rem; display: grid; grid-template-columns: repeat(auto-fill, minmax(17rem, 1fr)); gap: .4rem 1.2rem; font-size: .84rem; color: var(--ink-2); border-top: 1px solid var(--line); padding-top: .7rem; }
.hitos li { display: flex; gap: .5rem; align-items: flex-start; }
.hitos .n { flex: none; width: 20px; height: 20px; border-radius: 50%; background: var(--navy); color: #fff; font-size: .72rem; font-weight: 700; display: inline-flex; align-items: center; justify-content: center; margin-top: 1px; }
.hitos b { color: var(--ink); font-weight: 600; }

/* Barras apiladas (desgloses) */
.grid2 { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: .9rem; margin-top: .9rem; align-items: start; }
.grid2 .card { margin-top: 0; }
.bars { display: flex; flex-direction: column; gap: 2px; }
.brow { display: grid; grid-template-columns: minmax(7rem, 34%) 1fr 2.2rem; gap: .6rem; align-items: center; width: 100%; background: none; border: 0; border-radius: 4px; padding: .22rem .3rem; text-align: left; cursor: pointer; }
.brow:hover { background: #f3f7f8; }
.brow.sel { background: var(--teal-soft); box-shadow: inset 3px 0 0 var(--teal); }
.brow .bl { font-size: .86rem; line-height: 1.25; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
.brow.vacio .bl, .brow.vacio .bv { color: var(--muted); }
.brow .bt { height: 14px; display: flex; align-items: center; }
.brow .bf { display: flex; gap: 2px; height: 14px; }
.brow .seg { flex-basis: 0; min-width: 3px; }
.brow .seg:last-child { border-radius: 0 4px 4px 0; }
.brow .bv { font-weight: 700; font-size: .86rem; text-align: right; }
.f1 { background: var(--fav-1); } .f2 { background: var(--fav-2); } .f3 { background: var(--fav-3); }
.f4 { background: var(--fav-4); } .f5 { background: var(--fav-5); }

/* Vista de tabla alternativa */
.tabla-alt { overflow-x: auto; }
.tabla-alt table, .registro { border-collapse: collapse; width: 100%; font-size: .86rem; }
.tabla-alt th, .tabla-alt td { padding: .35rem .5rem; border-bottom: 1px solid var(--line); text-align: right; }
.tabla-alt th:first-child, .tabla-alt td:first-child { text-align: left; }
.tabla-alt thead th { color: var(--ink-2); font-weight: 600; border-bottom: 1px solid var(--axis); }
.tabla-alt td { font-variant-numeric: tabular-nums; }

/* Registro cronológico */
.tabla-wrap { margin-top: .8rem; background: var(--surface); border: 1px solid var(--line); border-radius: 8px; overflow-x: auto; }
.registro th { text-align: left; font-weight: 600; font-size: .8rem; color: var(--ink-2); background: #f7f9fa; padding: .55rem .6rem; border-bottom: 1px solid var(--axis); white-space: nowrap; }
.registro td { padding: .55rem .6rem; border-bottom: 1px solid var(--line); vertical-align: top; }
.registro tbody tr.fila:hover td { background: #fafcfc; }
.registro .fecha { white-space: nowrap; font-variant-numeric: tabular-nums; }
.registro .fecha small, .registro .cong small, .registro .prop small { display: block; color: var(--muted); font-size: .78rem; }
.registro .prop b { font-weight: 600; }
.registro .art { max-width: 20rem; }
.registro .exp { width: 2.4rem; padding-right: 0; }
.exp-b { width: 28px; height: 28px; border-radius: 6px; border: 1px solid #aeb8c2; background: #fff; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; }
.exp-b:hover { background: var(--teal-soft); border-color: var(--teal); }
.exp-b svg { transition: transform .15s; }
.exp-b[aria-expanded="true"] svg { transform: rotate(90deg); }
.estado-b { display: inline-flex; align-items: center; gap: .4rem; white-space: nowrap; }
.estado-b i { width: 10px; height: 10px; border-radius: 2px; flex: none; }
.vacio-fila td { text-align: center; padding: 1.6rem 1rem; color: var(--ink-2); }
.det td { background: #f7f9fa; padding: 1rem 1.1rem 1.2rem 3.2rem; }
.det-grid { display: grid; grid-template-columns: minmax(15rem, 22rem) 1fr; gap: 1.5rem; }
.det h4 { font-size: .88rem; margin: 0 0 .5rem; color: var(--navy); }
.paso-lista { list-style: none; padding: 0; margin: 0; }
.paso { position: relative; padding: 0 0 1rem 2rem; }
.paso:not(:last-child)::before { content: ""; position: absolute; left: 9px; top: 22px; bottom: 2px; width: 2px; background: #c4ced8; }
.paso .n { position: absolute; left: 0; top: 0; width: 20px; height: 20px; border-radius: 50%; background: var(--navy); color: #fff; font-size: .72rem; font-weight: 700; display: inline-flex; align-items: center; justify-content: center; }
.paso b { display: block; font-size: .88rem; }
.paso .d { font-size: .8rem; color: var(--muted); }
.paso .t { font-size: .82rem; color: var(--ink-2); margin-top: .1rem; }
.campos { display: grid; grid-template-columns: repeat(auto-fit, minmax(11rem, 1fr)); gap: .6rem 1.2rem; margin-bottom: .9rem; }
.campos dt { font-size: .76rem; color: var(--muted); }
.campos dd { margin: 0; font-size: .88rem; font-weight: 600; }
.texto-b { margin: .2rem 0 .6rem; }
.texto-b p { font-size: .88rem; max-width: 75ch; }
.texto-b .k { font-size: .76rem; color: var(--muted); display: block; margin-bottom: .05rem; }
.cpt { display: inline-flex; gap: .3rem; align-items: center; }
.det a { color: var(--teal-dark); }

/* Pie */
.pie { background: var(--navy); color: #c6d3df; padding: 1.4rem 0 2rem; font-size: .82rem; margin-top: 2rem; }
.pie .wrap { display: grid; gap: .6rem; max-width: 1240px; }
.pie p { max-width: 95ch; }
.pie strong { color: #fff; }

/* Tooltip */
.tip { position: fixed; z-index: 200; pointer-events: none; background: var(--navy); color: #fff; border-radius: 6px; padding: .5rem .65rem; font-size: .82rem; max-width: 19rem; box-shadow: 0 4px 14px rgba(16,42,67,.25); }
.tip[hidden] { display: none; }
.tip-t { font-weight: 600; margin-bottom: .3rem; line-height: 1.25; }
.tip-r { display: flex; align-items: center; gap: .45rem; line-height: 1.5; }
.tip-r .key { width: 12px; height: 3px; border-radius: 2px; flex: none; }
.tip-r b { min-width: 1.4rem; text-align: right; font-size: .9rem; }
.tip-r span { color: #d3deea; }
.tip-p { margin-top: .3rem; color: #b6c6d6; font-size: .78rem; line-height: 1.3; }

/* Responsive */
@media (min-width: 1000px) { .filtros-wrap { position: sticky; top: 0; } }
@media (max-width: 1100px) { .kpis { grid-template-columns: repeat(3, minmax(0, 1fr)); } .hero { grid-column: 1 / -1; min-height: 0; flex-direction: row; align-items: flex-end; flex-wrap: wrap; } .hero .s { border-top: 0; padding-top: 0; } }
@media (max-width: 900px) { .grid2 { grid-template-columns: minmax(0, 1fr); } .det-grid { grid-template-columns: minmax(0, 1fr); } .det td { padding-left: 1.1rem; } }
@media (max-width: 640px) {
  .masthead-in { flex-direction: column; gap: .8rem; }
  .kpis { grid-template-columns: repeat(2, minmax(0, 1fr)); }
  .f, .f select, #f-tipo, #f-entidad { width: 100%; }
  .f-fecha { flex: 1 1 8rem; } .f-fecha input { width: 100%; }
  .acciones { margin-left: 0; width: 100%; } .acciones .btn { flex: 1; }
  .brow { grid-template-columns: minmax(5.5rem, 38%) 1fr 2rem; }
  .segs button { padding: 0 .55rem; }
}

/* Alto contraste / impresión */
@media (forced-colors: active) {
  .seg, .f1, .f2, .f3, .f4, .f5 { forced-color-adjust: none; border: 1px solid CanvasText; }
  .tile { border: 2px solid CanvasText; }
}
@media (prefers-reduced-motion: reduce) { * { transition: none !important; } }
@media print {
  .filtros-wrap, .skip, .link, .acciones, .exp { display: none !important; }
  body { background: #fff; font-size: 12px; }
  .card, .tabla-wrap { break-inside: avoid; }
  .masthead, .pie, .hero { -webkit-print-color-adjust: exact; print-color-adjust: exact; }
  * { -webkit-print-color-adjust: exact; print-color-adjust: exact; }
}

In [ ]:
%%writefile tablero-pnd-2018-2022/assets/app.js
/* Tablero de proposiciones al PND 2018-2022 — lógica (JavaScript puro, sin dependencias) */
(function () {
  'use strict';

  const D = window.PND_DATA;
  if (!D || !Array.isArray(D.registros)) {
    document.getElementById('contenido').insertAdjacentHTML('afterbegin',
      '<p style="padding:1rem 0">No se pudieron cargar los datos. Verifique que exista <code>data/proposiciones.js</code> junto a <code>index.html</code>.</p>');
    return;
  }

  /* ------------------------------------------------------------------ utilidades */
  const R = D.registros;
  const ORDEN_FAV = D.meta.orden_favorabilidad;           // Favorable, Parcialmente favorable, Desfavorable, Pendiente, Retirada
  const FAV_IDX = Object.fromEntries(ORDEN_FAV.map((f, i) => [f, i + 1]));
  const FAV_VAR = (f) => `var(--fav-${FAV_IDX[f]})`;
  const MESES = ['ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic'];
  const DAY = 86400000;
  const NS = 'http://www.w3.org/2000/svg';
  const collator = new Intl.Collator('es');
  const nf = new Intl.NumberFormat('es-CO');
  const $ = (sel, root = document) => root.querySelector(sel);

  function setAttrs(e, a) {
    if (!a) return;
    for (const k in a) {
      const v = a[k];
      if (v == null || v === false) continue;
      if (k.startsWith('on')) e.addEventListener(k.slice(2), v);
      else e.setAttribute(k, v === true ? '' : v);
    }
  }
  function add(e, kids) {
    for (const k of kids.flat()) {
      if (k == null || k === false) continue;
      e.append(k.nodeType ? k : document.createTextNode(String(k)));   // textContent: los datos nunca se interpretan como HTML
    }
  }
  const h = (tag, attrs, ...kids) => { const e = document.createElement(tag); setAttrs(e, attrs); add(e, kids); return e; };
  const s = (tag, attrs, ...kids) => { const e = document.createElementNS(NS, tag); setAttrs(e, attrs); add(e, kids); return e; };

  const parseISO = (iso) => new Date(iso + 'T00:00:00Z');
  const isoDe = (ms) => new Date(ms).toISOString().slice(0, 10);
  const fecha = (iso) => { const d = parseISO(iso); return `${d.getUTCDate()} ${MESES[d.getUTCMonth()]} ${d.getUTCFullYear()}`; };
  const fechaCorta = (ms) => { const d = new Date(ms); return `${String(d.getUTCDate()).padStart(2, '0')} ${MESES[d.getUTCMonth()]}`; };
  const difDias = (a, b) => Math.round((parseISO(b) - parseISO(a)) / DAY);
  const lunesDe = (iso) => { const d = parseISO(iso); return d.getTime() - ((d.getUTCDay() + 6) % 7) * DAY; };
  const camaraCorta = (c) => (c.startsWith('Senado') ? 'Senado' : 'Cámara');

  function cop(n) {
    if (!n) return 'Sin impacto estimado';
    const a = Math.abs(n), sg = n < 0 ? '−' : '';
    if (a >= 1e12) return `${sg}$${(a / 1e12).toLocaleString('es-CO', { maximumFractionDigits: 1 })} billones`;
    if (a >= 1e9) return `${sg}$${nf.format(Math.round(a / 1e9))} mil millones`;
    return `${sg}$${nf.format(Math.round(a / 1e6))} millones`;
  }

  function contarFav(rows) {
    const c = Object.fromEntries(ORDEN_FAV.map((f) => [f, 0]));
    rows.forEach((r) => { c[r.favorabilidad] += 1; });
    return c;
  }

  /* ------------------------------------------------------------------ estado y filtros */
  const state = { congresista: '', partido: '', posicion: '', camara: '', entidad: '', tipo: '', fav: '', desde: '', hasta: '' };
  const ETIQUETA = { congresista: 'Congresista', partido: 'Partido', posicion: 'Gobierno / Oposición', camara: 'Cámara', entidad: 'Entidad', tipo: 'Tipo', fav: 'Favorabilidad', desde: 'Desde', hasta: 'Hasta' };
  const expandidos = new Set();
  const vistaTabla = new Set();

  function coincide(r, st) {
    return (!st.congresista || r.congresista === st.congresista) &&
      (!st.partido || r.partido === st.partido) &&
      (!st.posicion || r.posicion === st.posicion) &&
      (!st.camara || r.camara === st.camara) &&
      (!st.entidad || r.entidad === st.entidad) &&
      (!st.tipo || r.tipo === st.tipo) &&
      (!st.fav || r.favorabilidad === st.fav) &&
      (!st.desde || r.fecha >= st.desde) &&
      (!st.hasta || r.fecha <= st.hasta);
  }
  const filtrar = (st = state) => R.filter((r) => coincide(r, st));
  const ordenCrono = (a, b) => (a.fecha + a.hora + a.id).localeCompare(b.fecha + b.hora + b.id);

  function poner(clave, valor) { state[clave] = valor; render(); }
  function alternar(clave, valor) { poner(clave, state[clave] === valor ? '' : valor); }

  /* ------------------------------------------------------------------ dimensiones (desgloses) */
  const DIMS = [
    { clave: 'posicion', titulo: 'Posición frente al Gobierno', campo: 'posicion', fijo: ['Gobierno', 'Independiente', 'Oposición'] },
    { clave: 'camara', titulo: 'Cámara', campo: 'camara' },
    { clave: 'partido', titulo: 'Partido', campo: 'partido' },
    { clave: 'tipo', titulo: 'Tipo de proposición', campo: 'tipo' },
    { clave: 'entidad', titulo: 'Entidad líder del análisis', campo: 'entidad', etiqueta: 'entidad_corta' },
    { clave: 'congresista', titulo: 'Congresista', campo: 'congresista' },
  ];
  DIMS.forEach((d) => {
    const tot = {};
    R.forEach((r) => { tot[r[d.campo]] = (tot[r[d.campo]] || 0) + 1; });
    d.categorias = d.fijo || Object.keys(tot).sort((a, b) => tot[b] - tot[a] || collator.compare(a, b));
    d.maxGlobal = Math.max(...d.categorias.map((c) => tot[c] || 0), 1);
    d.etiquetaDe = (valor) => (d.etiqueta ? (R.find((r) => r[d.campo] === valor) || {})[d.etiqueta] || valor : valor);
  });

  /* ------------------------------------------------------------------ tooltip */
  const tip = $('#tip');
  function nodosTip(titulo, filas, pie) {
    return [h('div', { class: 'tip-t' }, titulo),
      ...filas.map((f) => h('div', { class: 'tip-r' }, h('i', { class: 'key', style: `background:${f.color}` }), h('b', {}, String(f.valor)), h('span', {}, f.etiqueta))),
      pie ? h('div', { class: 'tip-p' }, pie) : null];
  }
  function mostrarTip(nodos, x, y) {
    tip.replaceChildren(...nodos.filter(Boolean));
    tip.hidden = false;
    const w = tip.offsetWidth, hh = tip.offsetHeight;
    let left = x + 14, top = y + 14;
    if (left + w > window.innerWidth - 8) left = x - w - 14;
    if (top + hh > window.innerHeight - 8) top = y - hh - 14;
    tip.style.left = Math.max(8, left) + 'px';
    tip.style.top = Math.max(8, top) + 'px';
  }
  const ocultarTip = () => { tip.hidden = true; };
  function ligarTip(el, generar) {
    el.addEventListener('pointermove', (ev) => mostrarTip(generar(), ev.clientX, ev.clientY));
    el.addEventListener('pointerleave', ocultarTip);
    el.addEventListener('focus', () => { const b = el.getBoundingClientRect(); mostrarTip(generar(), b.left + b.width / 2, b.top); });
    el.addEventListener('blur', ocultarTip);
  }
  const filasFav = (counts) => ORDEN_FAV.filter((f) => counts[f] > 0).map((f) => ({ color: FAV_VAR(f), valor: counts[f], etiqueta: f }));

  /* ------------------------------------------------------------------ controles de filtro */
  function opcionesSelect(sel, valores, textoTodos) {
    sel.replaceChildren(h('option', { value: '' }, textoTodos), ...valores.map((v) => h('option', { value: v.valor }, v.texto)));
  }
  function contar(campo) { const t = {}; R.forEach((r) => { t[r[campo]] = (t[r[campo]] || 0) + 1; }); return t; }

  function iniciarControles() {
    const cCong = contar('congresista'), cPart = contar('partido'), cEnt = contar('entidad'), cTipo = contar('tipo');
    const partidoDe = Object.fromEntries(R.map((r) => [r.congresista, r.partido]));
    opcionesSelect($('#f-congresista'), Object.keys(cCong).sort(collator.compare).map((n) => ({ valor: n, texto: `${n} · ${partidoDe[n]} (${cCong[n]})` })), 'Todos los congresistas');
    opcionesSelect($('#f-partido'), Object.keys(cPart).sort(collator.compare).map((n) => ({ valor: n, texto: `${n} (${cPart[n]})` })), 'Todos los partidos');
    opcionesSelect($('#f-entidad'), Object.keys(cEnt).sort(collator.compare).map((n) => ({ valor: n, texto: `${n} (${cEnt[n]})` })), 'Todas las entidades');
    opcionesSelect($('#f-tipo'), Object.keys(cTipo).sort(collator.compare).map((n) => ({ valor: n, texto: `${n} (${cTipo[n]})` })), 'Todos los tipos');
    [['congresista', '#f-congresista'], ['partido', '#f-partido'], ['entidad', '#f-entidad'], ['tipo', '#f-tipo']]
      .forEach(([k, sel]) => $(sel).addEventListener('change', (e) => poner(k, e.target.value)));

    const segmentos = (idCont, clave, opciones) => {
      const cont = $(`${idCont} .segs`);
      opciones.forEach((o) => cont.append(h('button', { type: 'button', 'data-v': o.valor, onclick: () => poner(clave, o.valor) }, o.texto)));
    };
    segmentos('#f-posicion', 'posicion', [{ valor: '', texto: 'Todas' }, ...DIMS[0].fijo.map((v) => ({ valor: v, texto: v }))]);
    segmentos('#f-camara', 'camara', [{ valor: '', texto: 'Ambas' }, ...DIMS[1].categorias.map((v) => ({ valor: v, texto: camaraCorta(v) }))]);

    const fechas = R.map((r) => r.fecha).concat(D.hitos.map((x) => x.fecha)).sort();
    ['#f-desde', '#f-hasta'].forEach((id) => { $(id).min = fechas[0]; $(id).max = fechas[fechas.length - 1]; });
    $('#f-desde').addEventListener('change', (e) => poner('desde', e.target.value));
    $('#f-hasta').addEventListener('change', (e) => poner('hasta', e.target.value));
    $('#limpiar').addEventListener('click', () => { Object.keys(state).forEach((k) => { state[k] = ''; }); render(); });
    $('#exportar').addEventListener('click', exportarCSV);
  }

  function sincronizarControles() {
    $('#f-congresista').value = state.congresista;
    $('#f-partido').value = state.partido;
    $('#f-entidad').value = state.entidad;
    $('#f-tipo').value = state.tipo;
    $('#f-desde').value = state.desde;
    $('#f-hasta').value = state.hasta;
    ['posicion', 'camara'].forEach((k) => {
      document.querySelectorAll(`#f-${k} .segs button`).forEach((b) => b.setAttribute('aria-pressed', String(b.dataset.v === state[k])));
    });
  }

  function renderEstado(rows) {
    const est = $('#estado-filtros');
    est.replaceChildren('Mostrando ', h('strong', {}, String(rows.length)), ` de ${R.length} proposiciones`);
    const chips = $('#chips');
    chips.replaceChildren();
    Object.keys(state).filter((k) => state[k]).forEach((k) => {
      const valor = ['desde', 'hasta'].includes(k) ? fecha(state[k]) : state[k];
      chips.append(h('li', {}, h('button', { type: 'button', 'aria-label': `Quitar filtro ${ETIQUETA[k]}: ${valor}`, onclick: () => poner(k, '') },
        `${ETIQUETA[k]}: ${valor}`, h('span', { class: 'x', 'aria-hidden': 'true' }, '×'))));
    });
  }

  /* ------------------------------------------------------------------ KPI: conteo de favorabilidad */
  function renderKPIs(rows) {
    const sinFav = filtrar({ ...state, fav: '' });          // las categorías siguen visibles aunque se filtre por una
    const cuentas = contarFav(sinFav);
    const neto = rows.reduce((a, r) => a + r.impacto, 0);
    const cont = $('#kpis');
    cont.replaceChildren(
      h('div', { class: 'hero' },
        h('div', {}, h('div', { class: 'v' }, nf.format(rows.length)), h('div', { class: 'l' }, rows.length === 1 ? 'proposición en la selección' : 'proposiciones en la selección')),
        h('div', { class: 's' }, `de ${R.length} en el registro · impacto fiscal neto estimado: ${cop(neto)}`)),
      ...ORDEN_FAV.map((f) => {
        const pct = sinFav.length ? Math.round((cuentas[f] / sinFav.length) * 100) : 0;
        return h('button', { type: 'button', class: 'tile', style: `--c:${FAV_VAR(f)}`, 'aria-pressed': String(state.fav === f),
          'aria-label': `${f}: ${cuentas[f]} proposiciones. ${state.fav === f ? 'Quitar filtro' : 'Filtrar'}`, onclick: () => alternar('fav', f) },
          h('span', { class: 'v' }, String(cuentas[f])), h('span', { class: 'l' }, f), h('span', { class: 'p' }, sinFav.length ? `${pct} % de la selección` : 'Sin datos'));
      }));
  }

  /* ------------------------------------------------------------------ línea de tiempo */
  const hitos = D.hitos.map((x, i) => ({ ...x, n: i + 1, ms: parseISO(x.fecha).getTime() }));
  const todasFechas = R.map((r) => parseISO(r.fecha).getTime()).concat(hitos.map((x) => x.ms));
  const semIni = lunesDe(isoDe(Math.min(...todasFechas)));
  const semFin = lunesDe(isoDe(Math.max(...todasFechas))) + 7 * DAY;
  const semanas = []; for (let t = semIni; t < semFin; t += 7 * DAY) semanas.push(t);
  const porSemana = (rows) => { const m = new Map(semanas.map((t) => [t, []])); rows.forEach((r) => m.get(lunesDe(r.fecha)).push(r)); return m; };
  const maxSemGlobal = Math.max(...[...porSemana(R).values()].map((v) => v.length));
  const paso = maxSemGlobal <= 8 ? 2 : maxSemGlobal <= 20 ? 5 : 10;
  const yMax = Math.ceil(maxSemGlobal / paso) * paso;
  const rangoSemana = (t) => { const a = new Date(t), b = new Date(t + 6 * DAY); return `${a.getUTCDate()} ${MESES[a.getUTCMonth()]} – ${b.getUTCDate()} ${MESES[b.getUTCMonth()]} ${b.getUTCFullYear()}`; };

  function trazoSup(x, y, w, hh, r) {                       // barra con extremo superior redondeado, base recta
    const rr = Math.min(r, hh, w / 2);
    return `M${x},${y + hh}V${y + rr}Q${x},${y} ${x + rr},${y}H${x + w - rr}Q${x + w},${y} ${x + w},${y + rr}V${y + hh}Z`;
  }

  function renderTimeline(rows) {
    const cont = $('#timeline');
    cont.replaceChildren();
    const W = Math.max(cont.clientWidth || 960, 760), H = 300;
    const m = { l: 40, r: 14, t: 48, b: 40 };
    const pw = W - m.l - m.r, ph = H - m.t - m.b;
    const X = (ms) => m.l + ((ms - semIni) / (semFin - semIni)) * pw;
    const Y = (v) => m.t + ph - (v / yMax) * ph;
    const svg = s('svg', { width: W, height: H, viewBox: `0 0 ${W} ${H}`, role: 'group',
      'aria-label': 'Columnas apiladas: proposiciones recibidas por semana según favorabilidad, con los hitos del trámite' });

    for (let v = 0; v <= yMax; v += paso) {
      svg.append(s('line', { class: v === 0 ? 'base' : 'ygrid', x1: m.l, x2: W - m.r, y1: Y(v), y2: Y(v) }),
        s('text', { x: m.l - 8, y: Y(v) + 4, 'text-anchor': 'end' }, String(v)));
    }
    const wkPx = pw / semanas.length, cada = wkPx < 52 ? 2 : 1;
    semanas.forEach((t, i) => {
      svg.append(s('line', { class: 'base', x1: X(t), x2: X(t), y1: Y(0), y2: Y(0) + 4 }));
      if (i % cada === 0) svg.append(s('text', { x: X(t + 3.5 * DAY), y: Y(0) + 18, 'text-anchor': 'middle' }, fechaCorta(t)));
    });
    svg.append(s('text', { x: W - m.r, y: H - 6, 'text-anchor': 'end' }, 'Semanas, con inicio el lunes indicado'));

    hitos.forEach((x) => svg.append(s('line', { class: 'hito-l', x1: X(x.ms), x2: X(x.ms), y1: 36, y2: Y(0) })));

    const bw = Math.min(24, wkPx - 8);
    const rHito = Math.min(11, Math.max(8, (pw / ((semFin - semIni) / DAY)) * 1.5 - 0.5));
    const mapa = porSemana(rows);
    semanas.forEach((t) => {
      const rs = mapa.get(t), cuentas = contarFav(rs), total = rs.length;
      const iso0 = isoDe(t), iso1 = isoDe(t + 6 * DAY);
      const g = s('g', { class: 'col' + (state.desde === iso0 && state.hasta === iso1 ? ' sel' : ''),
        tabindex: total ? 0 : null, role: total ? 'button' : null,
        'aria-label': total ? `Semana del ${rangoSemana(t)}: ${total} proposiciones. Seleccionar para filtrar por fecha.` : null });
      g.append(s('rect', { class: 'wk-bg', x: X(t), y: m.t - 4, width: X(t + 7 * DAY) - X(t), height: ph + 4 }));
      let base = Y(0);
      const activos = ORDEN_FAV.filter((f) => cuentas[f] > 0);
      activos.forEach((f, i) => {
        const alto = (cuentas[f] / yMax) * ph;
        const ultimo = i === activos.length - 1, primero = i === 0;
        const y = base - alto, hh = alto - (primero ? 0 : 2), x = X(t + 3.5 * DAY) - bw / 2;
        g.append(ultimo ? s('path', { d: trazoSup(x, y, bw, hh, 4), fill: FAV_VAR(f) }) : s('rect', { x, y, width: bw, height: hh, fill: FAV_VAR(f) }));
        base = y;
      });
      if (total) {
        g.addEventListener('click', () => { if (state.desde === iso0 && state.hasta === iso1) { state.desde = ''; state.hasta = ''; render(); } else { state.desde = iso0; state.hasta = iso1; render(); } });
        g.addEventListener('keydown', (ev) => { if (ev.key === 'Enter' || ev.key === ' ') { ev.preventDefault(); g.dispatchEvent(new Event('click')); } });
        ligarTip(g, () => nodosTip(`Semana del ${rangoSemana(t)}`, [{ color: 'var(--axis)', valor: total, etiqueta: 'proposiciones recibidas' }, ...filasFav(cuentas)]));
        g.style.cursor = 'pointer';
      }
      svg.append(g);
    });

    hitos.forEach((x) => {
      const g = s('g', { class: 'hito-c', tabindex: 0, role: 'img', 'aria-label': `Hito ${x.n}: ${x.etiqueta}, ${fecha(x.fecha)}` });
      g.append(s('circle', { cx: X(x.ms), cy: 20, r: rHito, fill: 'var(--navy)', stroke: '#fff', 'stroke-width': 2 }),
        s('text', { class: 'num', x: X(x.ms), y: 24, 'text-anchor': 'middle' }, String(x.n)));
      ligarTip(g, () => nodosTip(`${x.n}. ${x.etiqueta}`, [], `${fecha(x.fecha)} — ${x.detalle}`));
      svg.append(g);
    });

    if (!rows.length) svg.append(s('text', { class: 'empty', x: W / 2, y: m.t + ph / 2, 'text-anchor': 'middle' }, 'Sin proposiciones para los filtros actuales'));
    cont.append(svg);
  }

  function renderTimelineTabla(rows) {
    const mapa = porSemana(rows);
    const t = h('table', {}, h('thead', {}, h('tr', {}, h('th', { scope: 'col' }, 'Semana'), ...ORDEN_FAV.map((f) => h('th', { scope: 'col' }, f)), h('th', { scope: 'col' }, 'Total'))),
      h('tbody', {}, ...semanas.map((w) => { const c = contarFav(mapa.get(w)); return h('tr', {}, h('th', { scope: 'row' }, rangoSemana(w)), ...ORDEN_FAV.map((f) => h('td', {}, String(c[f]))), h('td', {}, String(mapa.get(w).length))); })));
    $('#timeline-tabla').replaceChildren(t);
  }

  function iniciarHitos() {
    $('#hitos').replaceChildren(...hitos.map((x) => h('li', {}, h('span', { class: 'n' }, String(x.n)), h('span', {}, h('b', {}, `${fecha(x.fecha)} · ${x.etiqueta}. `), x.detalle))));
  }

  /* ------------------------------------------------------------------ registro cronológico */
  const chev = () => s('svg', { width: 14, height: 14, viewBox: '0 0 14 14', 'aria-hidden': 'true' }, s('path', { d: 'M5 2.5l4.5 4.5L5 11.5', fill: 'none', stroke: 'currentColor', 'stroke-width': 1.8, 'stroke-linecap': 'round', 'stroke-linejoin': 'round' }));

  function detalle(r) {
    const pasos = r.trazabilidad.map((p, i) => h('li', { class: 'paso' },
      h('span', { class: 'n', 'aria-hidden': 'true' }, String(i + 1)), h('b', {}, p.hito),
      h('span', { class: 'd' }, `${fecha(p.fecha)}${i ? (difDias(r.fecha, p.fecha) === 0 ? ' · mismo día' : ` · +${difDias(r.fecha, p.fecha)} días`) : ''}`),
      h('div', { class: 't' }, p.detalle)));
    const dato = (k, v) => [h('dt', {}, k), h('dd', {}, v)];
    const bloque = (k, v) => (v ? h('div', { class: 'texto-b' }, h('span', { class: 'k' }, k), h('p', {}, v)) : null);
    const enlace = /^https:\/\//.test(r.url) ? h('a', { href: r.url, target: '_blank', rel: 'noopener noreferrer' }, 'Documento de referencia del trámite (se abre en otra pestaña)') : null;
    return h('tr', { class: 'det', id: `det-${r.id}` }, h('td', { colspan: 8 },
      h('div', { class: 'det-grid' },
        h('div', {}, h('h4', {}, 'Trazabilidad'), h('ol', { class: 'paso-lista' }, pasos)),
        h('div', {},
          h('h4', {}, 'Contenido y análisis'),
          bloque('Texto original de la proposición', r.texto), bloque('Objeto / intención', r.objeto),
          bloque('Justificación presentada', r.justificacion), bloque('Observaciones', r.observaciones),
          h('dl', { class: 'campos' },
            ...dato('Eje líder', r.eje), ...dato('Tema', r.tema), ...dato('Circunscripción', r.circunscripcion), ...dato('Coponentes / apoyos', r.apoyos),
            ...dato('Recomendación del Gobierno', r.recomendacion), ...dato('Prioridad', r.prioridad), ...dato('Riesgo jurídico', r.riesgo),
            ...dato('Conceptos (técnico / jurídico / fiscal)', `${r.c_tecnico} / ${r.c_juridico} / ${r.c_fiscal}`),
            ...dato('Impacto fiscal estimado', cop(r.impacto)), ...dato('Entidad líder', r.entidad), ...dato('Responsable de seguimiento', r.responsable),
            ...dato('Fuente de la propuesta', r.fuente)),
          enlace))));
  }

  function renderRegistro(rows) {
    const tb = $('#registro-cuerpo');
    tb.replaceChildren();
    if (!rows.length) {
      tb.append(h('tr', { class: 'vacio-fila' }, h('td', { colspan: 8 }, 'Ninguna proposición coincide con los filtros actuales. Quite algún filtro o use «Limpiar filtros».')));
      return;
    }
    [...rows].sort(ordenCrono).forEach((r) => {
      const abierto = expandidos.has(r.id);
      tb.append(h('tr', { class: 'fila' },
        h('td', { class: 'exp' }, h('button', { type: 'button', class: 'exp-b', 'aria-expanded': String(abierto), 'aria-controls': `det-${r.id}`, 'aria-label': `${abierto ? 'Ocultar' : 'Ver'} trazabilidad de ${r.id}`,
          onclick: () => { if (expandidos.has(r.id)) expandidos.delete(r.id); else expandidos.add(r.id); renderRegistro(filtrar()); } }, chev())),
        h('td', { class: 'fecha' }, fecha(r.fecha), h('small', {}, `${r.hora} h`)),
        h('td', { class: 'prop' }, h('b', {}, r.id), h('small', {}, r.radicado)),
        h('td', { class: 'cong' }, r.congresista, h('small', {}, `${r.partido} · ${camaraCorta(r.camara)}`)),
        h('td', {}, r.posicion),
        h('td', { class: 'art' }, r.articulo),
        h('td', {}, r.tipo),
        h('td', {}, h('span', { class: 'estado-b' }, h('i', { class: `f${FAV_IDX[r.favorabilidad]}`, 'aria-hidden': 'true' }), r.estado))));
      if (abierto) tb.append(detalle(r));
    });
  }

  /* ------------------------------------------------------------------ desgloses apilados por favorabilidad */
  function iniciarDesgloses() {
    const grid = $('#desgloses-grid');
    DIMS.forEach((d) => {
      grid.append(h('article', { class: 'card', id: `card-${d.clave}` },
        h('header', { class: 'card-h' }, h('h3', {}, d.titulo),
          h('button', { type: 'button', class: 'link', 'aria-pressed': 'false', 'data-vista': d.clave, onclick: () => { if (vistaTabla.has(d.clave)) vistaTabla.delete(d.clave); else vistaTabla.add(d.clave); renderDesglose(d, filtrar()); } }, 'Ver tabla')),
        h('div', { class: 'bars', 'data-barras': d.clave }), h('div', { class: 'tabla-alt', 'data-tabla': d.clave, hidden: true })));
    });
  }

  function renderDesglose(d, rows) {
    const tabla = vistaTabla.has(d.clave);
    const card = $(`#card-${d.clave}`);
    const barras = $(`[data-barras="${d.clave}"]`, card), alt = $(`[data-tabla="${d.clave}"]`, card), btn = $('[data-vista]', card);
    btn.textContent = tabla ? 'Ver barras' : 'Ver tabla';
    btn.setAttribute('aria-pressed', String(tabla));
    barras.hidden = tabla; alt.hidden = !tabla;
    const porCat = new Map(d.categorias.map((c) => [c, []]));
    rows.forEach((r) => { if (porCat.has(r[d.campo])) porCat.get(r[d.campo]).push(r); });

    barras.replaceChildren(...d.categorias.map((c) => {
      const rs = porCat.get(c), cuentas = contarFav(rs), total = rs.length, sel = state[d.clave] === c, etiqueta = d.etiquetaDe(c);
      const fill = h('span', { class: 'bf', style: `width:${(total / d.maxGlobal) * 100}%` },
        ...ORDEN_FAV.filter((f) => cuentas[f] > 0).map((f) => h('span', { class: `seg f${FAV_IDX[f]}`, style: `flex-grow:${cuentas[f]}` })));
      const fila = h('button', { type: 'button', class: 'brow' + (sel ? ' sel' : '') + (total ? '' : ' vacio'), 'aria-pressed': String(sel),
        'aria-label': `${c}: ${total} proposiciones. ${sel ? 'Quitar filtro' : 'Filtrar'}`, onclick: () => alternar(d.clave, c) },
        h('span', { class: 'bl', title: c }, etiqueta), h('span', { class: 'bt' }, fill), h('span', { class: 'bv' }, String(total)));
      ligarTip(fila, () => nodosTip(c, total ? [{ color: 'var(--axis)', valor: total, etiqueta: 'proposiciones' }, ...filasFav(cuentas)] : [], total ? null : 'Sin proposiciones con los filtros actuales'));
      return fila;
    }));

    alt.replaceChildren(h('table', {},
      h('thead', {}, h('tr', {}, h('th', { scope: 'col' }, d.titulo), ...ORDEN_FAV.map((f) => h('th', { scope: 'col' }, f)), h('th', { scope: 'col' }, 'Total'))),
      h('tbody', {}, ...d.categorias.map((c) => { const cu = contarFav(porCat.get(c)); return h('tr', {}, h('th', { scope: 'row' }, c), ...ORDEN_FAV.map((f) => h('td', {}, String(cu[f]))), h('td', {}, String(porCat.get(c).length))); }))));
  }

  /* ------------------------------------------------------------------ exportación */
  function exportarCSV() {
    const rows = [...filtrar()].sort(ordenCrono);
    const cols = [['id', 'ID'], ['fecha', 'Fecha recepción'], ['hora', 'Hora'], ['congresista', 'Congresista'], ['camara', 'Cámara'], ['partido', 'Partido'], ['posicion', 'Posición frente al Gobierno'],
      ['circunscripcion', 'Circunscripción'], ['tipo', 'Tipo de proposición'], ['articulo', 'Artículo afectado'], ['eje', 'Eje líder'], ['tema', 'Tema'], ['estado', 'Estado'], ['favorabilidad', 'Favorabilidad'],
      ['recomendacion', 'Recomendación Gobierno'], ['prioridad', 'Prioridad'], ['riesgo', 'Riesgo jurídico'], ['impacto', 'Impacto fiscal estimado COP'], ['entidad', 'Entidad líder'],
      ['fecha_limite', 'Fecha límite respuesta'], ['ultima_actualizacion', 'Última actualización'], ['radicado', 'Radicado'], ['texto', 'Texto de la proposición']];
    const esc = (v) => { let t = String(v ?? ''); if (/^[=+\-@]/.test(t) && isNaN(Number(t))) t = "'" + t; return `"${t.replace(/"/g, '""')}"`; };
    const csv = '﻿' + [cols.map((c) => esc(c[1])).join(',')].concat(rows.map((r) => cols.map((c) => esc(r[c[0]])).join(','))).join('\r\n');
    const url = URL.createObjectURL(new Blob([csv], { type: 'text/csv;charset=utf-8' }));
    const a = h('a', { href: url, download: 'proposiciones_PND_filtradas.csv' });
    document.body.append(a); a.click(); a.remove();
    setTimeout(() => URL.revokeObjectURL(url), 1000);
  }

  /* ------------------------------------------------------------------ ciclo de render */
  function render() {
    ocultarTip();
    const rows = filtrar();
    sincronizarControles();
    renderEstado(rows);
    renderKPIs(rows);
    renderTimeline(rows);
    renderTimelineTabla(rows);
    renderRegistro(rows);
    DIMS.forEach((d) => renderDesglose(d, rows));
  }

  function iniciarLeyendas() {
    document.querySelectorAll('[data-legend]').forEach((el) => {
      el.replaceChildren(...ORDEN_FAV.map((f) => h('span', {}, h('i', { class: `f${FAV_IDX[f]}`, 'aria-hidden': 'true' }), f)));
    });
  }

  iniciarControles();
  iniciarLeyendas();
  iniciarHitos();
  iniciarDesgloses();
  $('#generado').textContent = `Datos generados el ${D.meta.generado} · ${D.meta.n_registros} registros simulados.`;
  $('#vista-timeline').addEventListener('click', (e) => {
    const t = $('#timeline-tabla').hidden;
    $('#timeline-tabla').hidden = !t; $('#timeline').hidden = t;
    e.currentTarget.textContent = t ? 'Ver gráfico' : 'Ver tabla';
    e.currentTarget.setAttribute('aria-pressed', String(t));
  });
  render();

  let anchoPrevio = $('#timeline').clientWidth, temporizador;
  window.addEventListener('resize', () => {
    clearTimeout(temporizador);
    temporizador = setTimeout(() => { const w = $('#timeline').clientWidth; if (w && w !== anchoPrevio) { anchoPrevio = w; renderTimeline(filtrar()); } }, 150);
  });
})();

In [ ]:
%%writefile tablero-pnd-2018-2022/README.md
# Tablero de proposiciones al Plan Nacional de Desarrollo 2018-2022

Tablero de control estático (HTML + CSS + JavaScript, sin dependencias externas) para el seguimiento de las
proposiciones presentadas por congresistas al **Plan Nacional de Desarrollo 2018-2022 «Pacto por Colombia, Pacto por
la Equidad»** (Ley 1955 de 2019; Proyecto de Ley 311/2019 Cámara – 227/2019 Senado).

> **Datos simulados.** Los 50 registros, los conceptos, los impactos fiscales y los radicados son simulados con fines
> demostrativos, y los nombres de los congresistas son ficticios. El calendario del trámite y la posición de los partidos
> frente al Gobierno (Estatuto de la Oposición, Ley 1909 de 2018) se basan en información pública. Este sitio no es un
> canal oficial del Congreso de la República ni del Departamento Nacional de Planeación.

## Qué contiene el tablero

| Elemento | Dónde |
|---|---|
| Trazabilidad cronológica | Gráfico semanal con 7 hitos del trámite + registro cronológico con la trazabilidad de cada proposición |
| Filtro por congresista | Barra de filtros y gráfico «Congresista» |
| Filtro por partido | Barra de filtros y gráfico «Partido» |
| Filtro Gobierno / Oposición / Independiente | Selector segmentado y gráfico «Posición frente al Gobierno» |
| Filtro por cámara | Selector segmentado y gráfico «Cámara» |
| Conteo de favorabilidad | Tarjetas superiores (clic para filtrar) y barras apiladas en todos los desgloses |
| Filtro por entidad | Barra de filtros y gráfico «Entidad líder del análisis» |
| Filtro por tipo de proposición | Barra de filtros y gráfico «Tipo de proposición» |
| Extras | Rango de fechas, chips de filtros activos, vista de tabla en cada gráfico, descarga CSV de la selección |

**Favorabilidad** (derivada de la columna *Estado*): Favorable = Acogida total o Aprobada · Parcialmente favorable =
Acogida parcial · Desfavorable = No acogida o Negada · Pendiente = En análisis · Retirada.

## Estructura del repositorio

```
├── index.html                 Página del tablero
├── assets/
│   ├── styles.css             Estilos
│   └── app.js                 Lógica (filtros, gráficos SVG, tabla, CSV)
├── data/
│   ├── proposiciones.js       Datos que lee la página (generado)
│   ├── proposiciones.json     Los mismos datos en JSON (generado)
│   └── Proposiciones_PND_2018-2022_diligenciado.xlsx   Cuadro fuente validado
├── notebooks/
│   └── generar_tablero_PND.ipynb   Notebook de Google Colab que valida el Excel y genera el sitio
├── INSTRUCCIONES_GITHUB.md
└── README.md
```

## Ver el tablero en el equipo

Abra `index.html` con doble clic (funciona sin servidor) o, desde la carpeta del proyecto:

```bash
python -m http.server 8000      # luego abra http://localhost:8000
```

## Actualizar los datos

1. Diligencie el Excel (hoja `Proposiciones (2)`; mantenga la hoja `Catalogos`, que alimenta las listas desplegables).
2. Abra `notebooks/generar_tablero_PND.ipynb` en [Google Colab](https://colab.research.google.com), ejecute todas las celdas y cargue el Excel cuando se le pida.
3. El notebook valida el cuadro (25 controles), regenera el sitio y descarga un ZIP. Reemplace en el repositorio la carpeta `data/` (o todo el contenido) y haga *commit*.

Controles de validación principales: campos obligatorios, IDs únicos, valores dentro de catálogos, fechas dentro del trámite
(6-feb a 25-may-2019), coherencia estado ↔ etapa del trámite, recomendación del Gobierno ↔ conceptos, congresista ↔ cámara/partido,
rangos del impacto fiscal.

## Publicar en GitHub Pages

Siga `INSTRUCCIONES_GITHUB.md`.

## Fuentes de contexto

- Ley 1955 de 2019 (Plan Nacional de Desarrollo 2018-2022).
- Gaceta del Congreso, trámite del Proyecto de Ley 311/2019 Cámara – 227/2019 Senado.
- Ley 1909 de 2018 (Estatuto de la Oposición) y declaraciones de los partidos frente al Gobierno 2018-2022.

In [ ]:
%%writefile tablero-pnd-2018-2022/INSTRUCCIONES_GITHUB.md
# Cómo publicar el tablero en su GitHub personal

Resultado: una página pública en `https://SU_USUARIO.github.io/tablero-pnd-2018-2022/`.
Reemplace `SU_USUARIO` por su nombre de usuario de GitHub.

## Opción A — Desde el navegador (sin instalar nada)

1. Inicie sesión en <https://github.com> (o cree una cuenta gratuita).
2. Cree el repositorio: **+ → New repository**.
   - Repository name: `tablero-pnd-2018-2022`
   - Visibilidad: **Public** (GitHub Pages es gratuito para repositorios públicos).
   - Deje sin marcar *Add a README*, *.gitignore* y *license* (el proyecto ya los trae).
   - Pulse **Create repository**.
3. En la página del repositorio vacío, pulse **uploading an existing file**.
4. En su computador, descomprima el ZIP del proyecto. Abra la carpeta y arrastre **su contenido** (no la carpeta contenedora):
   `index.html`, `assets/`, `data/`, `notebooks/`, `README.md`, `INSTRUCCIONES_GITHUB.md` y, si se muestran, `.gitignore` y `.nojekyll`.
   - Arrastrar carpetas funciona en Chrome, Edge y Firefox. Los archivos que empiezan por punto pueden estar ocultos en su sistema; no son indispensables.
5. Espere a que termine la carga y pulse **Commit changes** (mensaje sugerido: `Publicar tablero PND 2018-2022`).
6. Active la publicación: **Settings → Pages → Build and deployment**.
   - Source: **Deploy from a branch**
   - Branch: **main** · carpeta **/ (root)** → **Save**.
7. Espere 1 a 3 minutos y recargue la página de *Pages*: aparecerá la URL pública. Ábrala y verifique que el tablero carga con los 50 registros.

## Opción B — Con Git (terminal)

```bash
cd tablero-pnd-2018-2022            # carpeta descomprimida
git init -b main
git add .
git commit -m "Publicar tablero PND 2018-2022"
git remote add origin https://github.com/SU_USUARIO/tablero-pnd-2018-2022.git
git push -u origin main
```

GitHub pide autenticación: use un *Personal Access Token* (Settings → Developer settings → Personal access tokens) como contraseña,
o inicie sesión antes con GitHub CLI (`gh auth login`). Después active **Settings → Pages** como en el paso 6 de la opción A.

## Actualizar el tablero con nuevos datos

1. Edite el Excel `data/Proposiciones_PND_2018-2022_diligenciado.xlsx` (conserve la hoja `Catalogos`).
2. Abra `notebooks/generar_tablero_PND.ipynb` en <https://colab.research.google.com> (**Archivo → Subir cuaderno**), ejecute todas las celdas y cargue el Excel.
3. Descargue el ZIP que genera el notebook y reemplace en el repositorio los archivos de `data/`
   (**Add file → Upload files**, mismo nombre) y haga *commit*. La página se actualiza sola en 1-3 minutos.

## Si algo no funciona

| Síntoma | Causa probable y solución |
|---|---|
| Error 404 en la URL | Pages aún no termina de publicar (espere unos minutos) o el *Branch/folder* no es `main` / `(root)`. |
| Página sin datos o mensaje «No se pudieron cargar los datos» | Falta `data/proposiciones.js` o está fuera de la carpeta `data/`. Verifique la estructura. |
| Página sin estilos | Falta la carpeta `assets/` o se subió dentro de otra carpeta (el `index.html` debe quedar en la raíz del repositorio). |
| Cambios que no se ven | Pulse Ctrl+F5 (recarga sin caché). |

In [ ]:
(SITIO / ".nojekyll").write_text("", encoding="utf-8")
(SITIO / ".gitignore").write_text(".DS_Store\n__pycache__/\n.ipynb_checkpoints/\n", encoding="utf-8")
if Path("generar_tablero_PND.ipynb").exists():
    shutil.copy("generar_tablero_PND.ipynb", SITIO / "notebooks" / "generar_tablero_PND.ipynb")
print(*sorted(str(p.relative_to(SITIO)) for p in SITIO.rglob("*") if p.is_file()), sep="\n")

## 7. Vista previa en el cuaderno

In [ ]:
import html as _html
from IPython.display import HTML, display

def vista_previa(carpeta=SITIO, alto=900):
    base = Path(carpeta)
    doc = (base / "index.html").read_text(encoding="utf-8")
    doc = doc.replace('<link rel="stylesheet" href="assets/styles.css">', "<style>" + (base / "assets/styles.css").read_text(encoding="utf-8") + "</style>")
    doc = doc.replace('<script src="data/proposiciones.js"></script>', "<script>" + (base / "data/proposiciones.js").read_text(encoding="utf-8") + "</script>")
    doc = doc.replace('<script src="assets/app.js"></script>', "<script>" + (base / "assets/app.js").read_text(encoding="utf-8") + "</script>")
    display(HTML(f'<iframe srcdoc="{_html.escape(doc, quote=True)}" width="100%" height="{alto}" style="border:1px solid #c3c9cf;border-radius:6px"></iframe>'))

vista_previa()

## 8. Descargar el repositorio (ZIP)

In [ ]:
archivo_zip = shutil.make_archive("tablero-pnd-2018-2022", "zip", root_dir=".", base_dir=str(SITIO))
print("ZIP generado:", archivo_zip)
try:
    from google.colab import files
    files.download(archivo_zip)
except ImportError:
    pass

## 9. Publicar en GitHub

Resultado: una página pública en `https://SU_USUARIO.github.io/tablero-pnd-2018-2022/`.
Reemplace `SU_USUARIO` por su nombre de usuario de GitHub.

## Opción A — Desde el navegador (sin instalar nada)

1. Inicie sesión en <https://github.com> (o cree una cuenta gratuita).
2. Cree el repositorio: **+ → New repository**.
   - Repository name: `tablero-pnd-2018-2022`
   - Visibilidad: **Public** (GitHub Pages es gratuito para repositorios públicos).
   - Deje sin marcar *Add a README*, *.gitignore* y *license* (el proyecto ya los trae).
   - Pulse **Create repository**.
3. En la página del repositorio vacío, pulse **uploading an existing file**.
4. En su computador, descomprima el ZIP del proyecto. Abra la carpeta y arrastre **su contenido** (no la carpeta contenedora):
   `index.html`, `assets/`, `data/`, `notebooks/`, `README.md`, `INSTRUCCIONES_GITHUB.md` y, si se muestran, `.gitignore` y `.nojekyll`.
   - Arrastrar carpetas funciona en Chrome, Edge y Firefox. Los archivos que empiezan por punto pueden estar ocultos en su sistema; no son indispensables.
5. Espere a que termine la carga y pulse **Commit changes** (mensaje sugerido: `Publicar tablero PND 2018-2022`).
6. Active la publicación: **Settings → Pages → Build and deployment**.
   - Source: **Deploy from a branch**
   - Branch: **main** · carpeta **/ (root)** → **Save**.
7. Espere 1 a 3 minutos y recargue la página de *Pages*: aparecerá la URL pública. Ábrala y verifique que el tablero carga con los 50 registros.

## Opción B — Con Git (terminal)

```bash
cd tablero-pnd-2018-2022            # carpeta descomprimida
git init -b main
git add .
git commit -m "Publicar tablero PND 2018-2022"
git remote add origin https://github.com/SU_USUARIO/tablero-pnd-2018-2022.git
git push -u origin main
```

GitHub pide autenticación: use un *Personal Access Token* (Settings → Developer settings → Personal access tokens) como contraseña,
o inicie sesión antes con GitHub CLI (`gh auth login`). Después active **Settings → Pages** como en el paso 6 de la opción A.

## Actualizar el tablero con nuevos datos

1. Edite el Excel `data/Proposiciones_PND_2018-2022_diligenciado.xlsx` (conserve la hoja `Catalogos`).
2. Abra `notebooks/generar_tablero_PND.ipynb` en <https://colab.research.google.com> (**Archivo → Subir cuaderno**), ejecute todas las celdas y cargue el Excel.
3. Descargue el ZIP que genera el notebook y reemplace en el repositorio los archivos de `data/`
   (**Add file → Upload files**, mismo nombre) y haga *commit*. La página se actualiza sola en 1-3 minutos.

## Si algo no funciona

| Síntoma | Causa probable y solución |
|---|---|
| Error 404 en la URL | Pages aún no termina de publicar (espere unos minutos) o el *Branch/folder* no es `main` / `(root)`. |
| Página sin datos o mensaje «No se pudieron cargar los datos» | Falta `data/proposiciones.js` o está fuera de la carpeta `data/`. Verifique la estructura. |
| Página sin estilos | Falta la carpeta `assets/` o se subió dentro de otra carpeta (el `index.html` debe quedar en la raíz del repositorio). |
| Cambios que no se ven | Pulse Ctrl+F5 (recarga sin caché). |
